In [30]:
import json
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score




In [31]:

# Load the IMDb movie data
movies = [] 

with open("/Users/umarahmed/Downloads/imdb_movies_2000to2022.prolific (3).json", "r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            movies.append(json.loads(line))

print("Number of movies:", len(movies))


Number of movies: 20620


In [ ]:

good_movies = []

for movie in movies:
    if movie.get("title") and movie.get("actors") and movie.get("genres"):
        if isinstance(movie["genres"], list) and len(movie["genres"]) > 0:
            if isinstance(movie["genres"][0], str) and movie["genres"][0].strip():
                good_movies.append(movie)

movies = good_movies

print("Movies after cleaning:", len(movies))

Movies after cleaning: 20476


In [ ]:

actor_features = []
for movie in movies:
    actors = {}
    for actor in movie["actors"]:
        if len(actor) > 1 and actor[1]:
            actors[actor[1]] = 1
    actor_features.append(actors)

In [ ]:

movie_genres = [movie["genres"][0] for movie in movies]

In [ ]:

vectorizer = DictVectorizer()
X = vectorizer.fit_transform(actor_features)
y = movie_genres

In [ ]:

movie_numbers = list(range(len(movies)))
train_numbers, test_numbers = train_test_split(
    movie_numbers, test_size=0.2, random_state=42
)



X_train = X[train_numbers]
X_test = X[test_numbers]
y_train = [y[number] for number in train_numbers]
y_test = [y[number] for number in test_numbers]

In [ ]:

print("Number of movies:", len(movies))
print("Number of actor features:", X.shape[1])
print("Number of training movies:", len(train_numbers))
print("Number of testing movies:", len(test_numbers))
print("Missing actor features:", sum(len(actors) == 0 for actors in actor_features))
print("Missing genre labels:", sum(genre == "" for genre in y))

Number of movies: 20476
Number of actor features: 33057
Number of training movies: 16380
Number of testing movies: 4096
Missing actor features: 0
Missing genre labels: 0


In [ ]:

print("\nModel Results")
print("Depth | Training Accuracy | Testing Accuracy")

for depth in range(1, 11):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)

    train_score = accuracy_score(y_train, train_predictions)
    test_score = accuracy_score(y_test, test_predictions)

    print(f"{depth:5} | {train_score:.2%}            | {test_score:.2%}")



Model Results
Depth | Training Accuracy | Testing Accuracy
    1 | 24.29%            | 25.37%
    2 | 24.46%            | 25.66%
    3 | 24.59%            | 25.66%
    4 | 24.77%            | 25.85%
    5 | 24.90%            | 25.93%
    6 | 25.00%            | 26.10%
    7 | 25.21%            | 26.27%
    8 | 25.33%            | 26.49%
    9 | 25.46%            | 26.51%
   10 | 25.57%            | 26.54%


In [ ]:

model = DecisionTreeClassifier(max_depth=10, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print("\nFive Incorrect Predictions")
found = 0

for position, movie_number in enumerate(test_numbers):
    actual_genre = y_test[position]
    predicted_genre = predictions[position]

    if predicted_genre != actual_genre:
        print("Movie:", movies[movie_number]["title"])
        print("Actual main genre:", actual_genre)
        print("Predicted genre:", predicted_genre)
        print()
        found += 1

    if found == 5:
        break


Five Incorrect Predictions
Movie: Summer of the Flying Saucer
Actual main genre: Family
Predicted genre: Drama

Movie: The Anniversary Party
Actual main genre: Comedy
Predicted genre: Drama

Movie: Khakee
Actual main genre: Action
Predicted genre: Drama

Movie: The Collection
Actual main genre: Action
Predicted genre: Drama

Movie: Twin Sunsets
Actual main genre: Comedy
Predicted genre: Drama



In [ ]:

print("Number of different main genres:", len(set(y)))
print("Model used: Decision Tree Classifier") 
#

Number of different main genres: 22
Model used: Decision Tree Classifier
